In [ ]:
import serial
import time
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import random

PORT = "COM4"

class MonteCarloOptimizer:
    """
    Real-time Monte Carlo optimizer for finding optimal DAC inputs.
    Tests candidate solutions on actual hardware via Arduino.
    """

    def __init__(self, serial_port=PORT, baud_rate=115200):
        """
        Initialize the optimizer.

        Parameters:
        serial_port (str): Arduino COM port
        baud_rate (int): Serial communication baud rate
        """
        self.serial_port = serial_port
        self.baud_rate = baud_rate
        self.ser = None

        # DAC constraints
        self.num_dac_channels = 64
        self.min_voltage = 0.0
        self.max_voltage = 4.0
        self.voltage_step = 0.5
        self.valid_voltages = np.arange(self.min_voltage,
                                        self.max_voltage + self.voltage_step,
                                        self.voltage_step)

        # ADC configuration
        # CHANGE: Arduino returns 14 ADC values, but we filter to 10
        self.num_adc_channels_raw = 14  # Raw ADC channels from Arduino
        self.num_adc_channels_filtered = 10  # After removing indices 0, 2, 7, 11
        self.adc_indices_to_remove = [0, 2, 7, 11]  # Indices to filter out

    def connect_arduino(self):
        """Establish serial connection with Arduino."""
        try:
            self.ser = serial.Serial(self.serial_port, self.baud_rate, timeout=2)
            time.sleep(2)  # Wait for Arduino to reset
            print(f"Connected to Arduino on {self.serial_port}")
            return True
        except serial.SerialException as e:
            print(f"Failed to connect: {e}")
            return False

    def disconnect_arduino(self):
        """Close serial connection."""
        if self.ser and self.ser.is_open:
            # Reset DACs to 0V before disconnecting
            self.send_input_to_arduino(np.zeros(self.num_dac_channels))
            self.ser.close()
            print("Disconnected from Arduino")

    # NEW METHOD: Filter ADC output to remove specific indices
    def filter_adc_output(self, raw_adc_values):
        """
        Remove ADC values at indices 0, 2, 7, and 11.
        Converts 14 raw ADC values to 10 filtered values.

        Parameters:
        raw_adc_values (numpy.ndarray): 14 raw ADC values from Arduino

        Returns:
        numpy.ndarray: 10 filtered ADC values (indices 1, 3, 4, 5, 6, 8, 9, 10, 12, 13)
        """
        # Create a mask for indices to keep (all except 0, 2, 7, 11)
        all_indices = np.arange(self.num_adc_channels_raw)
        keep_mask = ~np.isin(all_indices, self.adc_indices_to_remove)

        # Filter the values
        filtered_values = raw_adc_values[keep_mask]

        return filtered_values

    def send_input_to_arduino(self, dac_values):
        """
        Send DAC input to Arduino and receive filtered ADC output.

        Parameters:
        dac_values (array): 64 DAC voltage values

        Returns:
        numpy.ndarray: 10 filtered ADC output values (after removing indices 0,2,7,11),
                      or None if failed
        """
        if not self.ser or not self.ser.is_open:
            print("Serial connection not open")
            return None

        # Format input as comma-separated string
        input_string = ",".join([f"{v:.1f}" for v in dac_values])

        try:
            # Send to Arduino
            self.ser.write((input_string + "\n").encode())

            # Wait for response
            start_time = time.time()
            while True:
                if self.ser.in_waiting > 0:
                    response = self.ser.readline().decode('utf-8').strip()

                    # Parse ADC values
                    adc_values_str = response.split(",")

                    # CHANGE: Expect 14 values from Arduino
                    if len(adc_values_str) == self.num_adc_channels_raw:
                        # Convert to numpy array
                        raw_adc_values = np.array([float(v) for v in adc_values_str])

                        #Filter to 10 values by removing indices 0, 2, 7, 11
                        filtered_adc_values = self.filter_adc_output(raw_adc_values)

                        return filtered_adc_values

                # Timeout after 3 seconds
                if time.time() - start_time > 3:
                    print("Timeout waiting for Arduino response")
                    return None

        except Exception as e:
            print(f"Error communicating with Arduino: {e}")
            return None

    def load_dataset(self, inputs_csv, outputs_csv):
        """
        Load the 100k dataset.

        Parameters:
        inputs_csv (str): Path to CSV with 64-value input strings
        outputs_csv (str): Path to CSV with 10 output values per row

        Returns:
        tuple: (inputs_df, outputs_array)
        """
        print("Loading dataset...")

        # Load outputs (10 columns for distance calculation)
        outputs_df = pd.read_csv(outputs_csv)
        if outputs_df.shape[1] != 10:
            raise ValueError(f"Expected 10 output columns, got {outputs_df.shape[1]}")

        # Load inputs
        inputs_df = pd.read_csv(inputs_csv)
        if inputs_df.shape[1] != 1:
            raise ValueError(f"Expected 1 input column, got {inputs_df.shape[1]}")

        print(f"Loaded {len(outputs_df)} datapoints")
        return inputs_df, outputs_df.values

    def parse_input_string(self, input_string):
        """
        Parse comma-separated input string to array of 64 floats.

        Parameters:
        input_string (str): "v1,v2,v3,...,v64"

        Returns:
        numpy.ndarray: 64 voltage values
        """
        values = input_string.strip().split(',')
        return np.array([float(v) for v in values])

    def find_k_nearest(self, outputs_array, desired_output, k=10):
        """
        Find k nearest neighbors to desired output.

        Parameters:
        outputs_array (numpy.ndarray): Dataset outputs (N x 10)
        desired_output (list): Desired 10-value output
        k (int): Number of neighbors to find

        Returns:
        numpy.ndarray: Indices of k nearest neighbors
        """
        print(f"Finding {k} nearest neighbors to desired output...")

        desired = np.array(desired_output).reshape(1, -1)
        distances = cdist(desired, outputs_array, metric='euclidean')[0]

        # Get k smallest distances
        k_indices = np.argpartition(distances, k)[:k]
        k_indices = k_indices[np.argsort(distances[k_indices])]

        print(f" ggs Found {k} nearest neighbors")
        print(f"  Distance range: {distances[k_indices[0]]:.4f} to {distances[k_indices[-1]]:.4f}")

        return k_indices

    def perturb_input(self, dac_values, perturbation_strength=1.0):
        """
        Create a perturbed version of DAC input.

        Parameters:
        dac_values (array): Original 64 DAC values
        perturbation_strength (float): How much to perturb (multiplier for step size)

        Returns:
        numpy.ndarray: Perturbed DAC values
        """
        perturbed = dac_values.copy()

        # Randomly perturb each channel
        for i in range(self.num_dac_channels):
            if random.random() < 0.3:  # 30% chance to perturb each channel
                # Choose perturbation: plus minus 0.5V or 1.0V
                change = random.choice([-1.0, -0.5, 0.5, 1.0]) * perturbation_strength
                new_value = perturbed[i] + change

                # Clip to valid range and round to nearest valid voltage
                new_value = np.clip(new_value, self.min_voltage, self.max_voltage)
                new_value = round(new_value / self.voltage_step) * self.voltage_step
                perturbed[i] = new_value

        return perturbed

    def calculate_fitness(self, measured_output, desired_output):
        """
        Calculate how good a measured output is (lower is better).

        Parameters:
        measured_output (array): 10 filtered ADC values
        desired_output (array): 10 desired output values

        Returns:
        float: Euclidean distance (fitness score)
        """
        # CHANGE: Now measured_output is already 10 values (filtered)
        # No need to slice - directly compare all 10 values
        return np.linalg.norm(measured_output - desired_output)

    def run_monte_carlo(self, inputs_df, outputs_array, desired_output,
                       k_start=10, max_iterations=100, candidates_per_input=5,
                       early_stop_threshold=0.01):
        """
        Run Monte Carlo optimization in real-time on hardware.

        Parameters:
        inputs_df (DataFrame): Input strings from dataset
        outputs_array (numpy.ndarray): Output values from dataset
        desired_output (list): Desired 10-value output
        k_start (int): Number of starting points from dataset
        max_iterations (int): Maximum iterations to run
        candidates_per_input (int): How many variations to test per iteration
        early_stop_threshold (float): Stop if fitness below this value

        Returns:
        dict: Best result with 'input', 'output', 'fitness', 'iteration'
        """

        print("STARTING MONTE CARLO OPTIMIZATION")

        print(f"Note: Arduino returns 14 ADC values")
        print(f"      Filtering out indices {self.adc_indices_to_remove}")
        print(f"      Using 10 filtered values for comparison")

        # Find k nearest starting points
        k_indices = self.find_k_nearest(outputs_array, desired_output, k=k_start)

        # Parse starting inputs
        starting_inputs = []
        for idx in k_indices:
            input_string = inputs_df.iloc[idx].values[0]
            dac_values = self.parse_input_string(input_string)
            starting_inputs.append(dac_values)

        print(f"\n Initialized with {k_start} starting points")

        # Track best solution
        best_solution = {
            'input': None,
            'output': None,
            'fitness': float('inf'),
            'iteration': 0
        }

        # Track all tested solutions to avoid retesting
        tested_solutions = set()

        # Main optimization loop
        for iteration in range(max_iterations):
            print(f"\n--- Iteration {iteration + 1}/{max_iterations} ---")

            iteration_candidates = []

            # Generate candidates from each starting point
            for i, base_input in enumerate(starting_inputs):
                for j in range(candidates_per_input):
                    # Perturb the input
                    candidate = self.perturb_input(base_input,
                                                   perturbation_strength=1.0/(iteration+1)**0.5)

                    # Create hashable representation to check if already tested
                    candidate_hash = tuple(candidate)
                    if candidate_hash in tested_solutions:
                        continue

                    tested_solutions.add(candidate_hash)
                    iteration_candidates.append(candidate)

            print(f"Testing {len(iteration_candidates)} new candidates...")

            # Test each candidate on hardware
            for idx, candidate in enumerate(iteration_candidates):
                # CHANGE: send_input_to_arduino now returns 10 filtered values
                measured_output = self.send_input_to_arduino(candidate)

                if measured_output is None:
                    continue

                # Verify we got 10 values
                if len(measured_output) != 10:
                    print(f"Expected 10 filtered values, got {len(measured_output)}")
                    continue

                # Calculate fitness
                fitness = self.calculate_fitness(measured_output, desired_output)

                # Update best if improved
                if fitness < best_solution['fitness']:
                    best_solution['input'] = candidate.copy()
                    best_solution['output'] = measured_output.copy()
                    best_solution['fitness'] = fitness
                    best_solution['iteration'] = iteration + 1

                    print(f" new best Fitness: {fitness:.4f}")

                # Progress update
                if (idx + 1) % 10 == 0:
                    print(f"  Progress: {idx + 1}/{len(iteration_candidates)} tested")

            print(f"Best fitness so far: {best_solution['fitness']:.4f}")

            # Early stopping
            if best_solution['fitness'] < early_stop_threshold:
                print(f"\n Reached target fitness so stopping early.")
                break

            # Update starting points for next iteration
            # Keep best performers as new starting points
            if iteration < max_iterations - 1:
                # Use current best as primary starting point
                starting_inputs = [best_solution['input']]

                # Add perturbed versions
                for _ in range(k_start - 1):
                    perturbed = self.perturb_input(best_solution['input'])
                    starting_inputs.append(perturbed)

        print(f"Best fitness: {best_solution['fitness']:.4f}")
        print(f"Found at iteration: {best_solution['iteration']}")
        print(f"Desired output (10 values): {desired_output}")
        print(f"Achieved output (10 values): {best_solution['output'].tolist()}")

        return best_solution

In [ ]:
if __name__ == "__main__":

    # Configuration
    SERIAL_PORT = PORT
    INPUTS_CSV = '100k_inputs.csv'
    OUTPUTS_CSV = '100k_outputs.csv'

    # CHANGE: Desired output is now explicitly 10 values
    # These 10 values correspond to ADC indices: 1, 3, 4, 5, 6, 8, 9, 10, 12, 13
    # (after removing indices 0, 2, 7, 11 from the raw 14 ADC readings)
    DESIRED_OUTPUT = [0.046, 0.081, 0.081, 0.005, 0.075, 0.011, 0.030, 0.008, 0.012, 0.028]

    # Monte Carlo parameters
    K_START = 10              # Number of starting points from dataset
    MAX_ITERATIONS = 50       # Maximum iterations
    CANDIDATES_PER_INPUT = 3  # Variations to test per starting point per iteration
    EARLY_STOP = 0.01         # Stop if fitness gets below this

    # Initialize optimizer
    optimizer = MonteCarloOptimizer(serial_port=SERIAL_PORT)

    try:
        # Connect to Arduino
        if not optimizer.connect_arduino():
            print("Failed to connect to Arduino. Exiting.")
            exit(1)

        # Load dataset
        inputs_df, outputs_array = optimizer.load_dataset(INPUTS_CSV, OUTPUTS_CSV)

        # Run optimization
        best_result = optimizer.run_monte_carlo(
            inputs_df,
            outputs_array,
            DESIRED_OUTPUT,
            k_start=K_START,
            max_iterations=MAX_ITERATIONS,
            candidates_per_input=CANDIDATES_PER_INPUT,
            early_stop_threshold=EARLY_STOP
        )

        # Save best result
        print("\n" + "="*60)
        print("SAVING RESULTS")
        print("="*60)

        # Save input as CSV
        input_str = ",".join([f"{v:.1f}" for v in best_result['input']])
        with open('best_input.txt', 'w') as f:
            f.write(input_str)
        print("Best input saved to: best_input.txt")

        # Save output (10 filtered values)
        output_str = ",".join([f"{v:.3f}" for v in best_result['output']])
        with open('best_output.txt', 'w') as f:
            f.write(output_str)
        print("Best output saved to: best_output.txt (10 filtered values)")

        # Save summary
        with open('optimization_summary.txt', 'w') as f:
            f.write(f"Summary\n")
            f.write(f"Best Fitness: {best_result['fitness']:.4f}\n")
            f.write(f"Iteration Found: {best_result['iteration']}\n")
            f.write(f"Desired Output (10 values): {DESIRED_OUTPUT}\n")
            f.write(f"Achieved Output (10 values): {best_result['output'].tolist()}\n")
            f.write(f"\nNote: ADC indices 0, 2, 7, 11 were filtered out\n")
            f.write(f"Output corresponds to ADC indices: 1, 3, 4, 5, 6, 8, 9, 10, 12, 13\n")
        print("Summary saved to: optimization_summary.txt")

    except KeyboardInterrupt:
        print("\n\n interrupted by user")

    except Exception as e:
        print(f"\n✗ Error: {e}")
        import traceback
        traceback.print_exc()

    finally:
        # Always disconnect properly
        optimizer.disconnect_arduino()
        print("\nDone!")